In [7]:
import torch
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report
from transformers import AutoTokenizer

from evaluate import evaluate_bert_bilstm_crf
from models.bert_bilstm_crf import BertBiLstmCrfNER
from utils.bert_bilstm_crf.data_utils import read_conll_2, NERDataset, build_collate_fn

In [8]:
MAX_LENGTH = 128
BATCH_SIZE = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_PATH = "../data/matscholar/test.txt"
MODEL_PATH = '../graph/model/bert_bilstm_crf/bert_bilstm_crf_iter_1.pt'

In [9]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

if checkpoint.get("architecture") != "tokenizer_bert_bilstm_crf":
    raise ValueError(
        "This checkpoint is not the sequential Tokenizer -> BERT -> BiLSTM -> CRF architecture. "
        "Retrain the model before testing."
    )

tag2idx = checkpoint["tag2idx"]
idx2tag = checkpoint.get("idx2tag")
if idx2tag is None:
    idx2tag = {v: k for k, v in tag2idx.items()}
else:
    idx2tag = {int(k): v for k, v in idx2tag.items()}

MODEL_NAME = checkpoint.get("model_name", "bert-base-cased")
BERT_HIDDEN_SIZE = checkpoint.get("bert_hidden_size", 768)
LSTM_HIDDEN_SIZE = checkpoint.get("lstm_hidden_size", 256)
DROPOUT = checkpoint.get("dropout", 0.25)

test_sentence, test_tags = read_conll_2(TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
test_dataset = NERDataset(test_sentence, test_tags)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    max_length=MAX_LENGTH,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

model = BertBiLstmCrfNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    lstm_hidden_size=LSTM_HIDDEN_SIZE,
    dropout=DROPOUT,
    id2label=idx2tag,
    label2id=tag2idx,
    expected_bert_hidden_size=BERT_HIDDEN_SIZE,
).to(DEVICE)

model.load_state_dict(checkpoint["model"])

test_loss, test_p, test_r, test_f1, y_true, y_pred = evaluate_bert_bilstm_crf(
    model, test_loader, idx2tag, DEVICE
)

print("\n========== Test Result ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall: {test_r:.4f}")
print(f"Test F1: {test_f1:.4f}")

print("\n========== Detailed Report ==========")
print(classification_report(y_true, y_pred, digits=4))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



========== Test Result ==========
Test Loss: 9.9798
Test Precision: 0.8418
Test Recall: 0.8652
Test F1: 0.8533

========== Detailed Report ==========
              precision    recall  f1-score   support

         APL     0.7347    0.7660    0.7500        94
         CMT     0.8632    0.8913    0.8770       184
         DSC     0.9020    0.8930    0.8975       402
         MAT     0.8694    0.9155    0.8918       698
         PRO     0.7921    0.8035    0.7978       621
         SMT     0.8289    0.8514    0.8400       222
         SPL     0.7660    0.8571    0.8090        42

   micro avg     0.8418    0.8652    0.8533      2263
   macro avg     0.8223    0.8540    0.8376      2263
weighted avg     0.8420    0.8652    0.8533      2263

